# Target shapes: a line, a curve, or a step?

When you know the response column, `target_shapes` helps choose how to
represent each numeric predictor. It compares the variance explained by a
straight line, a binned curve, and a single threshold. A strong nonlinear
shape can be useful even when Pearson correlation is small.

Use `profile(df, target="response")` to add the pass to the default
profile. With an explicit detector list, include `target_shapes`. The pass
uses all complete, finite predictor–target pairs, with at least 30 rows
and two distinct predictor values. It reports at most ten decisive shapes.

## The three comparisons

The linear score is squared Pearson correlation. For a binned curve,
observations are sorted by the predictor and assigned to at most 20
approximately equal-count bins, keeping ties together. Each bin predicts
its target mean. With total squared deviation `TSS`, bin residual sum of
squares `RSS`, sample size `n`, and bin budget `B`, the adjusted score is:

```text
curve_R2 = max(0, 1 - (RSS / TSS) * (n - 1) / (n - B))
```

The single-step score is the variance explained by two target means on
either side of the best split between distinct predictor values. The
reported `step_at` is the midpoint between the two bracketing values.
Spearman correlation adds a description of monotonicity.

A step is decisive when `step_r2 >= 0.05`, its improvement over the line is
at least 0.05, and it carries at least 80% of the curve's improvement over
the line. Otherwise, a curve is reported when `curve_r2 >= 0.10` and its
improvement over the line is at least 0.05. These are reporting thresholds,
not significance tests.

## Two examples

The first response jumps near a dose of four. The second follows a smooth
cubic curve. Each call below selects only the shape pass so the output
shows this comparison directly.


In [ ]:
import numpy as np
import polars as pl
import onceover

rng = np.random.default_rng(11)
n = 1_200
dose = rng.uniform(0, 10, n)
df_step = pl.DataFrame({
    "dose": dose,
    "response": np.where(dose < 4, 1.0, 3.0) + rng.normal(0, 0.1, n),
})
level = rng.uniform(-1, 1, n)
df_curve = pl.DataFrame({"level": level, "response": level ** 3 + rng.normal(0, 0.05, n)})

shape_reports = {
    "Dose step": onceover.profile(df_step, target="response", detectors=["target_shapes"]),
    "Smooth cubic": onceover.profile(df_curve, target="response", detectors=["target_shapes"]),
}
shape_rows = []
for label, report in shape_reports.items():
    for finding in report.select(category="dependencies", target="response"):
        metrics = finding["evidence"][0]["metrics"]
        shape_rows.append({"Example": label, "Form": finding["specification"]["form"],
                           "Linear R²": metrics["linear_r2"], "Curve R²": metrics["curve_r2"],
                           "Step R²": metrics["step_r2"], "Claim": finding["claim"]})
assert {row["Form"] for row in shape_rows} == {"step_shape", "curved_shape"}
from great_tables import GT, loc, style
(GT(pl.DataFrame(shape_rows))
 .fmt_number(columns=["Linear R²", "Curve R²", "Step R²"], decimals=3)
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


In [ ]:
import altair as alt

step_plot = (alt.Chart(df_step).mark_circle(size=13, opacity=0.25)
             .encode(x=alt.X("dose:Q", title="Dose"), y=alt.Y("response:Q", title="Response"))
             .properties(title="A threshold changes the level", width=330, height=230))
curve_plot = (alt.Chart(df_curve).mark_circle(size=13, opacity=0.25)
              .encode(x=alt.X("level:Q", title="Level"), y=alt.Y("response:Q", title="Response"))
              .properties(title="A smooth curve changes the slope", width=330, height=230))
(step_plot | curve_plot).interactive()


## Reading and using a shape finding

The statement is a `dependencies/predictive` finding with form `step_shape`
or `curved_shape`. Read `linear_r2`, `curve_r2`, `step_r2`, `curvature`,
`step_share`, `spearman`, and `n_used` in its evidence. A step also records
its cut. Linear and unrelated predictors can produce no shape finding
because neither adds a decisive shape interpretation.

These scores describe the same observations used to choose the shape;
they are not out-of-sample predictive performance. The pass is univariate:
it does not adjust for other predictors or discover joint interactions.
A step-shaped pattern can also be a steep smooth transition at the
available resolution. The [discontinuity screen](step_discontinuities.ipynb)
asks a different question using a smooth background, split-half evaluation,
and local gap comparisons. The [interaction pass](interactions.ipynb)
examines joint effects under a fitted target model.
